# Modelo Final - DeepLabV3+ para detección de deforestación

En este notebook se implementa la versión final del modelo de segmentación semántica utilizado para detectar zonas deforestadas mediante imágenes Sentinel-2 multitemporales.

La arquitectura final utilizada fue DeepLabV3+ con backbone ResNet50, entrenada sobre imágenes PRE y POST concatenadas.

In [ ]:
!pip install segmentation-models-pytorch

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
    confusion_matrix
)

import segmentation_models_pytorch as smp

In [15]:
tiles_path1 = "/content/drive/MyDrive/vision_sentinental_drive/tiles_par1"
labels_path1 = "/content/drive/MyDrive/vision_sentinental_drive/labels_par1"

tiles_path2 = "/content/drive/MyDrive/vision_sentinental_drive/tiles_par2"
labels_path2 = "/content/drive/MyDrive/vision_sentinental_drive/labels_par2"

tiles_path3 = "/content/drive/MyDrive/vision_sentinental_drive/tiles_par3"
labels_path3 = "/content/drive/MyDrive/vision_sentinental_drive/labels_par3"

In [11]:
class ChangeDetectionDataset(Dataset):

    def __init__(self, tiles_dirs, labels_dirs):

        self.samples = []

        # recorrer todas las carpetas
        for tiles_dir, labels_dir in zip(
            tiles_dirs,
            labels_dirs
        ):

            ids = sorted([
                f.split("_")[1]
                for f in os.listdir(tiles_dir)
                if "pre" in f
            ])

            for tile_id in ids:

                self.samples.append(
                    (tiles_dir, labels_dir, tile_id)
                )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        tiles_dir, labels_dir, tile_id = self.samples[idx]

        # =========================
        # CARGAR IMÁGENES PRE
        # =========================
        pre = np.load(
            os.path.join(
                tiles_dir,
                f"tile_{tile_id}_pre.npy"
            )
        )

        # =========================
        # CARGAR IMÁGENES POST
        # =========================
        post = np.load(
            os.path.join(
                tiles_dir,
                f"tile_{tile_id}_post.npy"
            )
        )

        # =========================
        # CONCATENAR 18 BANDAS
        # 9 PRE + 9 POST
        # =========================
        x = np.concatenate(
            [pre, post],
            axis=0
        )

        # =========================
        # CARGAR LABEL
        # =========================
        label = np.load(
            os.path.join(
                labels_dir,
                f"tile_{tile_id}_label.npy"
            )
        )

        # agregar canal
        label = np.expand_dims(
            label,
            axis=0
        )

        # =========================
        # CONVERTIR A TENSORES
        # =========================
        x = torch.tensor(
            x,
            dtype=torch.float32
        )

        label = torch.tensor(
            label,
            dtype=torch.float32
        )

        return x, label

In [16]:
tiles_dirs = [
    tiles_path1,
    tiles_path2,
    tiles_path3
]

labels_dirs = [
    labels_path1,
    labels_path2,
    labels_path3
]

dataset = ChangeDetectionDataset(
    tiles_dirs,
    labels_dirs
)

print("Total samples:", len(dataset))

Total samples: 48


In [17]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

In [18]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False
)

In [19]:
model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=18,
    classes=1
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [20]:
class DiceLoss(nn.Module):

    def __init__(self, smooth=1):

        super(DiceLoss, self).__init__()

        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.sigmoid(logits)

        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()

        dice = (
            2. * intersection + self.smooth
        ) / (
            probs.sum() + targets.sum() + self.smooth
        )

        return 1 - dice

In [21]:
def combined_loss(pred, target):

    bce = bce_loss(pred, target)

    dice = dice_loss(pred, target)

    return bce + dice

In [22]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

print("Optimizer y scheduler listos")

Optimizer y scheduler listos


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

cpu


In [26]:
pos_weight = torch.tensor([2.0]).to(device)

bce_loss = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

dice_loss = DiceLoss()

In [27]:
def combined_loss(pred, target):

    bce = bce_loss(pred, target)

    dice = dice_loss(pred, target)

    return bce + dice

In [28]:
x, y = next(iter(train_loader))

x = x.to(device)
y = y.to(device)

y_pred = model(x)

loss = combined_loss(y_pred, y)

print(loss)

tensor(1.7001, grad_fn=<AddBackward0>)


In [29]:
num_epochs = 5

train_losses = []
val_losses = []

best_val_loss = float("inf")

for epoch in range(num_epochs):

    # ======================
    # TRAIN
    # ======================
    model.train()

    train_loss = 0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred = model(x)

        loss = combined_loss(y_pred, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ======================
    # VALIDATION
    # ======================
    model.eval()

    val_loss = 0

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)
            y = y.to(device)

            y_pred = model(x)

            loss = combined_loss(y_pred, y)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    # scheduler
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # guardar mejor modelo
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.6f}"
    )

Epoch 1/5 | Train Loss: 1.0923 | Val Loss: 1.2590 | LR: 0.001000
Epoch 2/5 | Train Loss: 0.8425 | Val Loss: 0.7517 | LR: 0.001000
Epoch 3/5 | Train Loss: 0.7378 | Val Loss: 1.3597 | LR: 0.001000
Epoch 4/5 | Train Loss: 0.8426 | Val Loss: 0.8956 | LR: 0.001000
Epoch 5/5 | Train Loss: 0.7326 | Val Loss: 0.7525 | LR: 0.001000
